<a href="https://colab.research.google.com/github/yeab-samuel/commit-message-generator/blob/main/commit_message_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
   print("Hello, commit messages!")


Hello, commit messages!


In [2]:
   !pip install -q datasets

   from datasets import load_dataset

   data = load_dataset("Maxscha/commitbench", split="train", streaming=True)

   for row in data.take(2):
       print("MESSAGE:", row["message"])
       print("DIFF:", row["diff"][:500])
       print("-" * 40)

README.md:   0%|          | 0.00/5.67k [00:00<?, ?B/s]

MESSAGE: refactoring to use PhaseDiagram.pd_coords
DIFF: diff --git a/pymatgen/phasediagram/analyzer.py b/pymatgen/phasediagram/analyzer.py
index <HASH>..<HASH> 100644
--- a/pymatgen/phasediagram/analyzer.py
+++ b/pymatgen/phasediagram/analyzer.py
@@ -68,7 +68,7 @@ class PDAnalyzer(object):
         if set(comp.elements).difference(self._pd.elements):
             raise ValueError('{} has elements not in the phase diagram {}'
                              ''.format(comp, self._pd.elements))
-        c = [comp.get_atomic_fraction(e) for e in self._pd.e
----------------------------------------
MESSAGE: refactor(replset): reduce server count when destroyed
DIFF: diff --git a/lib/core/topologies/replset.js b/lib/core/topologies/replset.js
index <HASH>..<HASH> 100644
--- a/lib/core/topologies/replset.js
+++ b/lib/core/topologies/replset.js
@@ -300,15 +300,11 @@ function connectNewServers(self, servers, callback) {
 
       // Destroyed
       if (self.state === DESTROYED || self.state === U

In [3]:
   import json

   def is_python_only(diff):
       # find the "diff --git ..." lines = one per changed file
       files = [line.split(" b/")[-1] for line in diff.split("\n") if line.startswith("diff --git")]
       return len(files) == 1 and files[0].endswith(".py")

   data = load_dataset("Maxscha/commitbench", split="train", streaming=True)
   data = data.shuffle(seed=42, buffer_size=5000)

   examples = []
   for row in data:
       if is_python_only(row["diff"]) and len(row["diff"]) < 2000:
           examples.append({"diff": row["diff"], "message": row["message"]})
       if len(examples) == 300:
           break

   dev = examples[:100]
   test = examples[100:]

   json.dump(dev, open("dev.json", "w"))
   json.dump(test, open("test.json", "w"))

   print(len(dev), len(test))
   print("Example message:", examples[0]["message"])

100 200
Example message: Fix incorrect method call on StringIO in store.


In [6]:
   import torch
   from transformers import AutoTokenizer, AutoModelForCausalLM

   model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
   tokenizer = AutoTokenizer.from_pretrained(model_name)
   model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")

   def generate_message(diff):
       prompt = ("Write a one-line git commit message for this code change. "
                 "Reply with only the message.\n\n" + diff)
       chat = [{"role": "user", "content": prompt}]
       text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
       inputs = tokenizer(text, return_tensors="pt").to("cuda")
       output = model.generate(**inputs, max_new_tokens=40, do_sample=False)
       new_tokens = output[0][inputs["input_ids"].shape[1]:]
       return tokenizer.decode(new_tokens, skip_special_tokens=True).strip().split("\n")[0]

   for ex in dev[:3]:
       print("REAL:     ", ex["message"])
       print("GENERATED:", generate_message(ex["diff"]))
       print("-" * 40)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

REAL:      Fix incorrect method call on StringIO in store.
GENERATED: "Updated S3 download method to use getvalue instead of bytes."
----------------------------------------
REAL:      Renaming wrong named args to kwargs
GENERATED: Refactor fieldset initialization to use keyword arguments for clarity and flexibility.
----------------------------------------
REAL:      cleaned up code by removing commented out code
GENERATED: Fix broadcast address retrieval for non-AF_INET interfaces
----------------------------------------


In [7]:
   !pip install -q rouge-score
   from rouge_score import rouge_scorer

   scorer = rouge_scorer.RougeScorer(["rougeL"])

   def baseline_message(diff):
       first_line = diff.split("\n")[0]      # looks like: diff --git a/x.py b/x.py
       filename = first_line.split(" b/")[-1]
       return "Update " + filename

   def average_score(examples, predict):
       scores = []
       for ex in examples:
           prediction = predict(ex["diff"])
           scores.append(scorer.score(ex["message"], prediction)["rougeL"].fmeasure)
       return sum(scores) / len(scores)

   print("Baseline:", average_score(dev, baseline_message))
   print("Model   :", average_score(dev, generate_message))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Baseline: 0.07253453867864286
Model   : 0.14615451572861446


In [8]:
   def generate_with(instruction):
       def predict(diff):
           prompt = instruction + "\n\n" + diff
           chat = [{"role": "user", "content": prompt}]
           text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
           inputs = tokenizer(text, return_tensors="pt").to("cuda")
           output = model.generate(**inputs, max_new_tokens=40, do_sample=False)
           new_tokens = output[0][inputs["input_ids"].shape[1]:]
           message = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().split("\n")[0]
           return message.strip("\"'` ")
       return predict

   short_prompt = ("Write a git commit message for this code change: one short line, "
                   "at most 10 words, starting with a verb like Fix, Add, Remove, Rename or Update. "
                   "Describe what the change does, not what the code is about. "
                   "Reply with only the message.")

   print("Short prompt:", average_score(dev, generate_with(short_prompt)))

Short prompt: 0.13364437944413418


In [9]:
   results = []
   for ex in test:
       base = baseline_message(ex["diff"])
       pred = generate_message(ex["diff"])
       results.append({
           "diff": ex["diff"],
           "real": ex["message"],
           "baseline": base,
           "model": pred,
           "baseline_score": scorer.score(ex["message"], base)["rougeL"].fmeasure,
           "model_score": scorer.score(ex["message"], pred)["rougeL"].fmeasure,
       })

   json.dump(results, open("results.json", "w"), indent=2)

   n = len(results)
   print("TEST baseline:", sum(r["baseline_score"] for r in results) / n)
   print("TEST model   :", sum(r["model_score"] for r in results) / n)

TEST baseline: 0.07568542840779463
TEST model   : 0.14814146455598542


In [11]:
   from google.colab import drive
   drive.mount("/content/drive", force_remount=True)

   import os, shutil
   os.makedirs("/content/drive/MyDrive/commit_project", exist_ok=True)
   for f in ["dev.json", "test.json", "results.json"]:
       if os.path.exists(f):
           shutil.copy(f, "/content/drive/MyDrive/commit_project/" + f)
   print(os.listdir("/content/drive/MyDrive/commit_project"))

Mounted at /content/drive
['dev.json', 'test.json', 'results.json']
